# Exploring ERA5 Reanalysis Temperature Data over CONUS

**Course:** Climate and Climate Change  
**Topic:** Working with gridded reanalysis data and spatial visualization

---

## Learning Objectives

By the end of this exercise, you will be able to:
1. Understand what **ERA5 reanalysis** data is and why it's used in climate science
2. Open and inspect a **NetCDF** file using `xarray`
3. Extract and slice spatial data by region and time
4. Create a publication-quality **filled contour map** of surface temperature
5. Interpret spatial patterns in temperature across the continental United States (CONUS)

---

## Background

### What is ERA5?

[ERA5](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) is the fifth generation **global atmospheric reanalysis** produced by the European Centre for Medium-Range Weather Forecasts (ECMWF). Reanalysis data combines historical observations (weather stations, radiosondes, satellites) with a numerical weather model to produce a consistent, gridded record of the atmosphere dating back to 1940.

ERA5 provides:
- Hourly data at ~31 km horizontal resolution (~0.25°)
- Hundreds of atmospheric and surface variables
- Global coverage from 1940 to near-present

It is one of the most widely used datasets in climate research.

### What is NetCDF?

**NetCDF** (Network Common Data Form) is the standard file format for storing multidimensional scientific data (e.g., temperature varying in latitude, longitude, pressure level, and time). Nearly all climate and atmospheric datasets use this format.

---

## Step 1: Set Up the Environment

We first install and import the libraries we need. If you are running this on **Google Colab**, the cell below will install any missing packages automatically.

In [ ]:
# Install required packages (only needed on Colab or fresh environments)
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["xarray", "netCDF4", "cartopy", "matplotlib", "numpy", "gdown"]
for pkg in required:
    install(pkg)

print("All packages ready!")

In [ ]:
# Core scientific libraries
import numpy as np
import xarray as xr

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Cartopy for map projections and geographic features
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Suppress minor warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

---

## Step 2: Download the Dataset from Google Drive

The ERA5 dataset for this exercise is hosted on Google Drive. We use the `gdown` library to download it directly into our Colab session.

In [ ]:
import gdown
import os

# -------------------------------------------------------
# The variable below holds the file ID for the google file
# which the gdown package will download from Google Drive
GDRIVE_FILE_ID = "1XtK7DYuIPBKlhCte3gtHC900gFEWzUEk"
# -------------------------------------------------------

OUTPUT_FILENAME = "era5_conus_t2m.nc"

if not os.path.exists(OUTPUT_FILENAME):
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
    print(f"Downloading dataset from Google Drive...")
    gdown.download(url, OUTPUT_FILENAME, quiet=False)
    print(f"Download complete: {OUTPUT_FILENAME}")
else:
    print(f"File already exists: {OUTPUT_FILENAME}")

---

## Step 3: Open and Inspect the Dataset

We use `xarray` to open the NetCDF file. `xarray` is the standard Python library for working with labeled, multidimensional arrays — think of it as pandas for gridded scientific data.

In [ ]:
# Open the NetCDF file with xarray
ds = xr.open_dataset(OUTPUT_FILENAME)

# Print a summary of the dataset
print(ds)

### 🔍 Understanding the Dataset Structure

Take a moment to read the output above. Identify:
- **Dimensions**: What axes does the data have? (e.g., `time`, `latitude`, `longitude`)
- **Coordinates**: What are the actual values along each dimension?
- **Data variables**: What variables are stored? (We expect `t2m` = 2-meter air temperature)
- **Attributes**: Metadata like units, source, and creation date

> **❓ Question 1:** What are the latitude and longitude bounds of this dataset? Does it cover all of CONUS?

In [ ]:
# Inspect the 2-meter temperature variable specifically
t2m = ds['t2m']  # ERA5 variable name for 2-meter air temperature
print(t2m)
print(f"\nUnits: {t2m.attrs.get('units', 'not specified')}")
print(f"Long name: {t2m.attrs.get('long_name', 'not specified')}")
print(f"Shape: {t2m.shape}")
print(f"Dimensions: {t2m.dims}")

In [ ]:
# Check the time dimension
print("Available time steps:")
print(ds.time.values)

---

## Step 4: Prepare the Data for Plotting

ERA5 stores temperature in **Kelvin**. We'll convert to **Celsius** for more intuitive interpretation. We'll also select a single time step to plot.

In [ ]:
# Select the first time step (modify the index to explore other times)
time_idx = 0
selected_time = ds.time.values[time_idx]
print(f"Plotting data for: {selected_time}")

# Select that time step and convert Kelvin → Celsius
t2m_slice = ds['t2m'].isel(time=time_idx) - 273.15
t2m_slice.attrs['units'] = '°C'

# Pull out coordinate arrays for convenience
lons = ds['longitude'].values
lats = ds['latitude'].values

print(f"Temperature range: {float(t2m_slice.min()):.1f} to {float(t2m_slice.max()):.1f} °C")

> **❓ Question 2:** What is the temperature range across CONUS for your selected time step? Does the range make physical sense given the season?

---

## Step 5: Plot the Spatial Temperature Field

Now we create a filled contour map using `matplotlib` and `cartopy`. Cartopy handles the map projection and geographic features (coastlines, state borders, etc.).

In [ ]:
# --- Plot Configuration ---
# Map projection: LambertConformal is standard for CONUS display
proj = ccrs.LambertConformal(
    central_longitude=-96.0,
    central_latitude=37.5,
    standard_parallels=(33, 45)
)

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': proj}
)

# Set map extent to CONUS [west, east, south, north]
ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

# --- Filled Contour Plot ---
# Define contour levels (every 2°C from -30 to 40°C)
levels = np.arange(-30, 42, 2)

cf = ax.contourf(
    lons, lats, t2m_slice.values,
    levels=levels,
    cmap='RdBu_r',          # Diverging colormap: blue=cold, red=warm
    transform=ccrs.PlateCarree(),  # Data is in lat/lon, not projected
    extend='both'           # Extend colorbar for out-of-range values
)

# --- Contour Lines (optional overlay) ---
cl = ax.contour(
    lons, lats, t2m_slice.values,
    levels=levels[::5],     # Only every 5th level for readability
    colors='k',
    linewidths=0.5,
    transform=ccrs.PlateCarree()
)
ax.clabel(cl, inline=True, fontsize=7, fmt='%d°C')

# --- Geographic Features ---
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.8, linestyle='-')
ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray')
ax.add_feature(cfeature.OCEAN, color='lightcyan', zorder=0)
ax.add_feature(cfeature.LAKES, color='lightcyan', zorder=0)

# --- Gridlines ---
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.5,
    color='gray',
    alpha=0.5,
    linestyle='--'
)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = mticker.FixedLocator([-120, -110, -100, -90, -80, -70])
gl.ylocator = mticker.FixedLocator([25, 30, 35, 40, 45, 50])

# --- Colorbar ---
cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.04, shrink=0.8)
cbar.set_label('2-meter Air Temperature (°C)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

# --- Title ---
time_str = str(selected_time)[:16]  # Trim to YYYY-MM-DDTHH:MM
ax.set_title(
    f'ERA5 2-Meter Air Temperature over CONUS\n{time_str} UTC',
    fontsize=13, fontweight='bold', pad=10
)

plt.tight_layout()
plt.savefig('era5_t2m_conus.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as era5_t2m_conus.png")

---

## Step 6: Explore the Data Further

### 6a. Plot a Different Time Step

Modify `time_idx` in Step 4 and re-run Steps 4 and 5 to visualize temperature at a different time.

> **❓ Question 3:** How does the temperature pattern change between your two selected times? What meteorological processes might explain these differences?

### 6b. Compute and Plot the Time-Mean Temperature

Instead of a single snapshot, let's compute the **temporal mean** across all time steps.

In [ ]:
# Compute time mean
t2m_mean = ds['t2m'].mean(dim='time') - 273.15
t2m_mean.attrs['units'] = '°C'

print(f"Time-mean temperature range: {float(t2m_mean.min()):.1f} to {float(t2m_mean.max()):.1f} °C")

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 7), subplot_kw={'projection': proj})
ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

cf = ax.contourf(
    lons, lats, t2m_mean.values,
    levels=np.arange(-30, 42, 2),
    cmap='RdBu_r',
    transform=ccrs.PlateCarree(),
    extend='both'
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray')
ax.add_feature(cfeature.OCEAN, color='lightcyan', zorder=0)

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.04, shrink=0.8)
cbar.set_label('2-meter Air Temperature (°C)', fontsize=11)

n_times = len(ds.time)
ax.set_title(
    f'ERA5 Time-Mean 2-Meter Temperature over CONUS\n(mean over {n_times} time steps)',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('era5_t2m_mean_conus.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Step 7: Zonal Mean Temperature Profile

A **zonal mean** averages over all longitudes at each latitude, revealing the meridional temperature gradient — one of the fundamental drivers of atmospheric circulation.

In [ ]:
# Zonal (longitude) mean of the time-mean temperature
zonal_mean = t2m_mean.mean(dim='longitude')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(zonal_mean.values, lats, color='firebrick', linewidth=2)
ax.axvline(0, color='k', linewidth=0.8, linestyle='--', label='0°C')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('Zonal Mean 2-m Temperature vs. Latitude (CONUS domain)', fontsize=11)
ax.grid(True, alpha=0.4)
ax.legend()

plt.tight_layout()
plt.savefig('era5_zonal_mean.png', dpi=150, bbox_inches='tight')
plt.show()

> **❓ Question 4:** Describe the relationship between latitude and temperature shown in the zonal mean plot. What is the approximate temperature gradient (°C per degree of latitude) across CONUS? How does this relate to the driving of mid-latitude weather systems?

---

## Summary

In this exercise you:
- Downloaded a real **ERA5 reanalysis** NetCDF dataset
- Inspected its structure using `xarray`
- Converted units and selected spatial/temporal slices
- Created a **filled contour map** with state/country borders using `cartopy`
- Computed time means and a zonal mean temperature profile

These are core skills used daily in climate research. The same workflow applies to any gridded atmospheric variable (precipitation, wind, humidity, etc.) from any reanalysis or climate model output.

---

## 📝 Assignment Questions

Answer the following in a short write-up (1–2 paragraphs each):

1. Describe the large-scale temperature pattern you observe over CONUS. What physical factors (latitude, topography, proximity to ocean) seem to control the spatial distribution?

2. How does the snapshot (single time step) compare to the time-mean map? What does this tell us about **weather variability** vs. **climatological mean state**?

3. ERA5 is a **reanalysis**, not direct observations. What are the advantages and potential limitations of using reanalysis data in climate studies?

4. *(Bonus)* Modify the notebook to plot a different ERA5 variable (e.g., `sp` for surface pressure, or `u10`/`v10` for 10-m winds if available in the dataset). Describe what you find.

---

## Additional Resources

- [ERA5 documentation (ECMWF)](https://confluence.ecmwf.int/display/CKB/ERA5)
- [xarray documentation](https://docs.xarray.dev/)
- [Cartopy documentation](https://scitools.org.uk/cartopy/docs/latest/)
- [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/) — where you can download ERA5 data yourself

In [ ]:
# Clean up: close the dataset
ds.close()
print("Dataset closed. Notebook complete!")